# 🎤 Kịch bản Demo 10–15 phút

**Mục tiêu:** mỗi phút demo phải chứng minh một tiêu chí chấm điểm.

| Tiêu chí | Trọng số | Ai trình bày | Phút |
|---|---|---|---|
| Architecture & Justification | 25% | TV1, TV2 | 1:00–3:30 · 14:00–15:00 |
| Deployment & Automation | 20% | TV1 | 10:30–12:00 |
| Security & Correctness | 15% | TV4 | 6:00–8:30 |
| Reliability & Observability | 15% | TV3 | 8:30–10:30 · 12:00–13:00 |
| Application Completeness | 15% | TV4 | 3:30–6:00 |
| Cost & Communication | 10% | TV1 + cả nhóm | 13:00–14:00 |

**Cách dùng notebook:** đọc phần chuẩn bị → chạy cell **Kiểm tra trước demo** ở T−30 phút → bám theo từng đoạn timeline.

---
## ⏰ Chuẩn bị (T−30 phút, Đợt 3; chỉ TV1 chạy `terraform apply`)

- [ ] `terraform apply` xong, ghi lại `api_https_url` và URL Static Web App
- [ ] Chạy seed script (TV2): 3 org × 5 user × 20 task
- [ ] Mở sẵn các tab:
  - Web App: cửa sổ thường đăng nhập **Owner A**, cửa sổ ẩn danh đăng nhập **User X**
  - Azure Portal: Resource Group, PostgreSQL *Networking*, Storage container, Dashboard, Alert
  - Hộp thư nhận Alert · GitHub Actions
- [ ] Lấy token mới cho `tests/security/tokens.local.json` (token hết hạn sau ~60–90 phút)
- [ ] Terminal mở ở thư mục repo, phóng to cỡ chữ
- [ ] **Dự phòng:** video Rollback, ảnh email Alert, thư mục evidence của Đợt 2

### ✅ Kiểm tra trước demo
Cell dưới kiểm tra: đủ 4 token chưa, token còn bao nhiêu phút (đọc `exp` trong JWT, **không in token**), API có sống không.
Cấu hình qua biến môi trường `API_BASE_URL` hoặc sửa trực tiếp trong cell.

In [1]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists() or (p / "infra").is_dir())
print("Repo root:", ROOT.name)

Repo root: multi-tenant-project-management


In [2]:
import base64, json, os, time
import requests

API_BASE_URL = os.environ.get("API_BASE_URL", "")   # ví dụ "https://app-mtpm-saas-dev-api-xxxx.azurewebsites.net"
REQUIRED = ["TOKEN_A_OWNER", "TOKEN_A_GUEST", "TOKEN_X", "TOKEN_B_OWNER"]

tok_file = ROOT / "tests" / "security" / "tokens.local.json"
tokens = json.loads(tok_file.read_text(encoding="utf-8")) if tok_file.exists() else {}
tokens.update({k: os.environ[k] for k in REQUIRED if os.environ.get(k)})


def minutes_left(jwt: str):
    try:
        payload = jwt.split(".")[1]
        payload += "=" * (-len(payload) % 4)
        return (json.loads(base64.urlsafe_b64decode(payload))["exp"] - time.time()) / 60
    except Exception:
        return None


print("Token:")
for k in REQUIRED:
    if k not in tokens:
        print(f"  ❌ {k:14} chưa có")
        continue
    m = minutes_left(tokens[k])
    status = "⚠️ không đọc được exp" if m is None else ("❌ ĐÃ HẾT HẠN" if m <= 0 else ("⚠️ sắp hết" if m < 20 else "✅"))
    print(f"  {status} {k:14} còn {m:.0f} phút" if m is not None else f"  {status} {k}")

print("\nAPI:")
if not API_BASE_URL:
    print("  ⏳ Chưa đặt API_BASE_URL (bỏ qua khi chưa bật Azure)")
else:
    for path in ("/actuator/health", "/health"):
        try:
            r = requests.get(API_BASE_URL.rstrip("/") + path, timeout=10)
            print(f"  {'✅' if r.ok else '❌'} GET {path} → {r.status_code}")
            break
        except requests.RequestException as e:
            print(f"  ❌ GET {path} → {type(e).__name__}")

Token:
  ❌ TOKEN_A_OWNER  chưa có
  ❌ TOKEN_A_GUEST  chưa có
  ❌ TOKEN_X        chưa có
  ❌ TOKEN_B_OWNER  chưa có

API:
  ⏳ Chưa đặt API_BASE_URL (bỏ qua khi chưa bật Azure)


---
## 🕐 Dòng thời gian

### 0:00–1:00 · TV4: Mở đầu
- Bài toán: SaaS quản lý dự án cho nhiều tổ chức.
- Nguyên tắc sống còn: **không rò rỉ dữ liệu giữa tenant**, kể cả khi sửa ID hay giả mạo job.
- Thao tác: Slide 1.

### 1:00–3:30 · TV1: Kiến trúc & ranh giới mạng *(25%)*

| Nói | Cho xem trên Portal |
|---|---|
| VNet 2 subnet, DB không có đường từ Internet | PostgreSQL → *Networking* → **Public access: Disabled** |
| Blob private, chỉ tải qua SAS ngắn hạn | Storage → container `exports` → Access level **Private** |
| Không secret trong code: Managed Identity + Key Vault | App Service → *Identity* (System assigned: On) · *Configuration* (Key Vault reference) |

### 3:30–6:00 · TV4: Ứng dụng *(15%)*

1. Đăng nhập Entra External ID → **Chuyển Org** (dropdown trên cùng).
2. Task Board: tạo task, chuyển cột, giao người.
3. Members: mời thành viên, đổi role.
4. Audit: chỉ cột `request_id` (dùng để tra log).
5. Export CSV → đợi `COMPLETED` → **Tải CSV** (SAS hết hạn sau 5 phút; nếu kịp, mở lại link cũ → `403`).

### 6:00–8:30 · TV4: Bảo mật Test 1–3 chạy trực tiếp *(15%)*

**a) Chạy bộ test trên API thật**
```powershell
.\tests\security\run.ps1 -Target azure -BaseUrl <api_https_url>
```
→ mở `SUMMARY.md` → mở một file `.http.txt`: request của Org A gọi task Org B → **404**; token đã được che.

**b) Test 2 bằng UI** (dễ hiểu với người chấm)
Owner A thu hồi User X → cửa sổ ẩn danh của X bấm **F5** → thông báo `403 — Không phải thành viên… (request_id …)`.
👉 Nhấn mạnh: *JWT của X vẫn còn hạn, nhưng bị chặn vì API kiểm tra bảng memberships mỗi request.*

Cell dưới chạy bộ test từ notebook (để `RUN = True` khi đang demo):

In [3]:
import subprocess, sys
RUN = False   # đổi thành True khi demo trên Azure (cần API_BASE_URL + token hợp lệ)

if RUN and API_BASE_URL:
    env = {**os.environ, "API_TARGET": "azure", "API_BASE_URL": API_BASE_URL, "PYTHONIOENCODING": "utf-8"}
    res = subprocess.run([sys.executable, "-m", "pytest", "-q", "--no-header"], cwd=ROOT / "tests" / "security",
                         env=env, capture_output=True, text=True, encoding="utf-8")
    print(res.stdout[-2500:])
else:
    print("⏸️ Bỏ qua (RUN = False hoặc chưa có API_BASE_URL)")

⏸️ Bỏ qua (RUN = False hoặc chưa có API_BASE_URL)


### 8:30–10:30 · TV3: Failure Drill *(15%)*
1. Chạy script đẩy job có `claimedTenantId` giả mạo vào queue.
2. App Insights (KQL) → log JSON có `jobId`, lý do từ chối.
3. Metric tenant-mismatch tăng → **email Alert** đến hộp thư (~1–2 phút).
4. Song song: web app vẫn hoạt động bình thường (API không bị ảnh hưởng).
5. Dashboard 4 biểu đồ.

### 10:30–12:00 · TV1: CI/CD & Rollback *(20%)*
Commit nhỏ → GitHub Actions chạy → (dự phòng: video) → rollback về bản trước → chỉ version trên endpoint health.

### 12:00–13:00 · TV3: Load test k6
Ảnh kết quả k6 + biểu đồ latency: p95 = ⟦…⟧ ms, error rate = ⟦…⟧ %.

### 13:00–14:00 · TV1: Chi phí & Cleanup *(10%)*
Bảng cost estimate · Cost Management thực tế · Cost Alert $25/$50/$75 · số dư credit.

### 14:00–15:00 · TV2: Data model, RLS, connection pool, scaling
Slide ERD + policy RLS.

> 🧹 **Ngay sau khi chấm xong:** TV1 chạy `terraform destroy`, rồi chụp Resource Group trống.

---
## ❓ Câu hỏi phản biện thường gặp

| Câu hỏi | Ai trả lời | Ý chính (chi tiết: `DESIGN_REPORT.ipynb` §7) |
|---|---|---|
| Nếu sửa ID trên URL thì sao? | TV4 | 3 lớp: interceptor membership → truy vấn có `tenant_id` → RLS. Demo bằng Test 1 |
| Sao thu hồi có hiệu lực ngay dù JWT còn hạn? | TV2 | JWT chỉ là danh tính; quyền đọc từ `memberships` mỗi request |
| Message trong queue bị sửa? | TV3 | Worker coi message là không đáng tin, đối chiếu `export_jobs` |
| Tải tăng 10×? | TV1/TV2 | API stateless scale ngang; DB B1ms là nút cổ chai → pool, PgBouncer |
| Message bị giao 2 lần? | TV3 | Claim nguyên tử, blob path theo jobId, xóa message sau commit |
| Sao không dùng Functions/Container Apps? | TV1 | Bảng Alternative Compute |
| Có lọt token trong log không? | TV3/TV4 | Logback redact + `scan_secrets.py` trước khi nộp |